# Notebook 12 - Final hybrid GT_OK experiment

Notebook này kiểm tra giả thuyết rút ra từ notebook 11: V3/SAM có recall cao nhưng precision thấp, V5/color-prior cân bằng hơn nhưng vẫn còn phân đoạn thừa. Vì vậy phiên bản final không train lại từ đầu; nó thử nghiệm hậu xử lý/fusion có kiểm soát trên checkpoint V2, V3, V5.

Thiết kế đánh giá:

- Dùng 80 ảnh GT_OK có mặt nạ thủ công.
- Mỗi lớp chia 10 ảnh đầu làm `tune`, 10 ảnh còn lại làm `holdout`.
- Tune ngưỡng theo lớp và tùy chọn giới hạn coverage trên `tune`.
- Chọn mô hình final bằng IoU trên `holdout`, sau đó báo cáo thêm full GT_OK như chỉ số chẩn đoán.

Cách này chưa thay thế một thí nghiệm huấn luyện đầy đủ, nhưng đủ để kiểm chứng nhanh liệu hướng final có cải thiện thật so với V1/V2/V3/V5 hay chỉ là nhận xét lý thuyết.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.final_hybrid_gtok_experiment import run


In [2]:
payload = run()
best = payload["best_by_holdout_iou"]
hold = payload["results"][best]["holdout"]["overall"]
full = payload["results"][best]["full"]["overall"]

print("Best final candidate:", best)
print(f"Holdout: IoU={hold['iou']:.4f}, Dice={hold['dice']:.4f}, Precision={hold['precision']:.4f}, Recall={hold['recall']:.4f}")
print(f"Full GT_OK: IoU={full['iou']:.4f}, Dice={full['dice']:.4f}, Precision={full['precision']:.4f}, Recall={full['recall']:.4f}")
print("Metrics JSON: notebooks/models/segmentation_final/metrics/final_hybrid_gtok_metrics.json")
print("Summary CSV: notebooks/models/segmentation_final/metrics/final_hybrid_summary.csv")
print("Predictions: notebooks/models/segmentation_final/predictions/")


Best final candidate: FINAL_V2_V3_V5_fusion
Holdout: IoU=0.3076, Dice=0.4705, Precision=0.4311, Recall=0.5178
Full GT_OK: IoU=0.3083, Dice=0.4713, Precision=0.4128, Recall=0.5492
Metrics JSON: notebooks/models/segmentation_final/metrics/final_hybrid_gtok_metrics.json
Summary CSV: notebooks/models/segmentation_final/metrics/final_hybrid_summary.csv
Predictions: notebooks/models/segmentation_final/predictions/


In [3]:
import json
import pandas as pd
from pathlib import Path

metrics_path = ROOT / "notebooks/models/segmentation_final/metrics/final_hybrid_gtok_metrics.json"
summary_path = ROOT / "notebooks/models/segmentation_final/metrics/final_hybrid_summary.csv"
data = json.loads(metrics_path.read_text(encoding="utf-8"))
summary = pd.read_csv(summary_path)

display(summary.sort_values(["split", "iou"], ascending=[True, False]))


,method,split,iou,dice,precision,recall,pixel_accuracy,tp,fp,fn,tn
9,FINAL_V2_V3_V5_fusion,full,0.308333,0.471337,0.412792,0.549232,0.927818,129163.0,183738.0,106007.0,3595172.0
7,FINAL_V2_V5_fusion_30_70,full,0.289199,0.448649,0.417193,0.485236,0.930128,114113.0,159413.0,121057.0,3619497.0
11,FINAL_V2_V5_consensus_min,full,0.275261,0.431694,0.405763,0.461164,0.928864,108452.0,158827.0,126718.0,3620083.0
5,FINAL_V5_threshold,full,0.272861,0.428736,0.375890,0.498873,0.922113,117320.0,194793.0,117850.0,3584117.0
3,V5,full,0.218252,0.358303,0.235821,0.745508,NaN,NaN,NaN,NaN,NaN
0,V1,full,0.216889,0.356464,0.245974,0.647170,NaN,NaN,NaN,NaN,NaN
1,V2,full,0.197158,0.329376,0.245899,0.498661,NaN,NaN,NaN,NaN,NaN
2,V3,full,0.135138,0.238100,0.140874,0.768470,NaN,NaN,NaN,NaN,NaN
8,FINAL_V2_V3_V5_fusion,holdout,0.307593,0.470472,0.431073,0.517798,0.928506,63744.0,84129.0,59362.0,1799805.0
6,FINAL_V2_V5_fusion_30_70,holdout,0.284304,0.442736,0.434866,0.450896,0.930378,55508.0,72136.0,67598.0,1811798.0


In [4]:
best = data["best_by_holdout_iou"]
params = data["results"][best]["params"]
pd.DataFrame([
    {"class_name": cls, **values}
    for cls, values in params.items()
])


,class_name,threshold,cap,iou
0,ALGAL_LEAF_SPOT,0.40,0.03,0.155664
1,ALLOCARIDARA_ATTACK,0.65,0.18,0.406809
2,LEAF_BLIGHT,0.70,NaN,0.269638
3,PHOMOPSIS_LEAF_SPOT,0.90,0.05,0.092377


## Nhận xét sau khi chạy

Ô dưới đây tự sinh nhận xét từ kết quả thực nghiệm. Khi dùng trong luận văn/paper, cần ghi rõ đây là thực nghiệm final dạng hậu xử lý trên checkpoint đã có, không phải một mô hình được huấn luyện lại end-to-end.


In [5]:
baseline = data["baseline_metrics_full_gtok"]
best = data["best_by_holdout_iou"]
best_hold = data["results"][best]["holdout"]["overall"]
best_full = data["results"][best]["full"]["overall"]
v5_full = baseline["V5"]
v1_full = baseline["V1"]

delta_v5 = best_full["iou"] - v5_full["iou"]
delta_v1 = best_full["iou"] - v1_full["iou"]

print(f"Phiên bản final tốt nhất theo holdout là {best}.")
print(f"Trên holdout 40 ảnh: IoU={best_hold['iou']:.4f}, Dice={best_hold['dice']:.4f}, Precision={best_hold['precision']:.4f}, Recall={best_hold['recall']:.4f}.")
print(f"Trên full GT_OK 80 ảnh: IoU={best_full['iou']:.4f}, Dice={best_full['dice']:.4f}, Precision={best_full['precision']:.4f}, Recall={best_full['recall']:.4f}.")
print(f"So với V5 gốc, chênh lệch full GT_OK IoU là {delta_v5:+.4f}; so với V1 baseline là {delta_v1:+.4f}.")
if delta_v5 > 0 and delta_v1 > 0:
    print("Kết quả ủng hộ hướng final: kiểm soát ngưỡng/coverage theo lớp giúp giảm phân đoạn thừa và cải thiện IoU so với checkpoint trước đó.")
elif delta_v5 > 0:
    print("Kết quả cải thiện so với V5 nhưng chưa vượt rõ baseline V1; nên trình bày như bằng chứng định hướng, không phải kết luận mạnh.")
else:
    print("Kết quả chưa cải thiện so với V5; nhận xét quan trọng là cần thêm nhãn thủ công hoặc train lại với pseudo-label được kiểm soát tốt hơn.")


Phiên bản final tốt nhất theo holdout là FINAL_V2_V3_V5_fusion.
Trên holdout 40 ảnh: IoU=0.3076, Dice=0.4705, Precision=0.4311, Recall=0.5178.
Trên full GT_OK 80 ảnh: IoU=0.3083, Dice=0.4713, Precision=0.4128, Recall=0.5492.
So với V5 gốc, chênh lệch full GT_OK IoU là +0.0901; so với V1 baseline là +0.0914.
Kết quả ủng hộ hướng final: kiểm soát ngưỡng/coverage theo lớp giúp giảm phân đoạn thừa và cải thiện IoU so với checkpoint trước đó.
